# Week 5 PCA/SVD Component Interpretation

This notebook inspects the saved Week 5 dimensionality-reduction artifacts. It complements the plots by reading the actual PCA loadings, SVD top terms, reconstruction summaries, and dimensionality comparison tables.

The notebook is intentionally interpretation-only:

- no PCA fitting,
- no TruncatedSVD fitting,
- no clustering,
- no recommendation generation,
- no matrix/model mutation.

The goal is to help teammates understand what the reduced dimensions appear to represent and how cautiously those interpretations should be used in the report.

## 1. Load artifacts

The code below loads CSV and JSON outputs from `artifacts/week5/pca_svd`. Missing files produce warnings instead of invented values. If a table is missing, downstream cells skip the corresponding interpretation.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

CWD = Path.cwd()
if (CWD / "artifacts/week5/pca_svd").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "artifacts/week5/pca_svd").exists():
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD
    warnings.warn("Could not locate artifacts/week5/pca_svd from the current working directory. Check PROJECT_ROOT.")

ARTIFACT_DIR = PROJECT_ROOT / "artifacts/week5/pca_svd"
NOTES_OUT = ARTIFACT_DIR / "component_interpretation_notes.md"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

paths = {
    "pca_top_loadings": ARTIFACT_DIR / "pca_top_loadings.csv",
    "pca_component_loadings": ARTIFACT_DIR / "pca_component_loadings.csv",
    "pca_explained_variance": ARTIFACT_DIR / "pca_explained_variance.csv",
    "pca_reconstruction_metrics": ARTIFACT_DIR / "pca_reconstruction_metrics.csv",
    "svd_top_terms": ARTIFACT_DIR / "svd_top_terms.csv",
    "svd_component_sweep": ARTIFACT_DIR / "svd_component_sweep.csv",
    "svd_explained_variance_selected": ARTIFACT_DIR / "svd_explained_variance_selected.csv",
    "dimensionality_comparison": ARTIFACT_DIR / "dimensionality_comparison.csv",
    "pca_selected_config": ARTIFACT_DIR / "pca_selected_config.json",
    "svd_selected_config": ARTIFACT_DIR / "svd_selected_config.json",
    "reduction_pipeline_config": ARTIFACT_DIR / "reduction_pipeline_config.json",
}

def load_csv_safe(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        warnings.warn(f"Missing CSV artifact: {path}")
        return None
    return pd.read_csv(path)

def load_json_safe(path: Path) -> dict:
    if not path.exists():
        warnings.warn(f"Missing JSON artifact: {path}")
        return {}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

csvs = {name: load_csv_safe(path) for name, path in paths.items() if path.suffix == ".csv"}
configs = {name: load_json_safe(path) for name, path in paths.items() if path.suffix == ".json"}

print("Loaded CSV artifacts:")
for name, df in csvs.items():
    print(f"- {name}: {'missing' if df is None else df.shape}")

print("\nLoaded JSON artifacts:")
for name, cfg in configs.items():
    print(f"- {name}: {'missing/empty' if not cfg else 'loaded'}")

PROJECT_ROOT = /Users/kanyewest/Documents/recipe_recommendation_project
Loaded CSV artifacts:
- pca_top_loadings: (150, 6)
- pca_component_loadings: (225, 5)
- pca_explained_variance: (15, 4)
- pca_reconstruction_metrics: (15, 4)
- svd_top_terms: (6000, 7)
- svd_component_sweep: (4, 10)
- svd_explained_variance_selected: (200, 4)
- dimensionality_comparison: (5, 14)

Loaded JSON artifacts:
- pca_selected_config: loaded
- svd_selected_config: loaded
- reduction_pipeline_config: loaded


## 2. PCA variance and reconstruction summary

PCA was applied to the dense, scaled numeric matrix. The central interpretation questions are:

- How many components were selected?
- How much numeric variance did they preserve?
- How quickly does reconstruction error decrease?
- Which original numeric features dominate each component?

In [2]:
pca_cfg = configs.get("pca_selected_config", {})
pca_var = csvs.get("pca_explained_variance")
pca_recon = csvs.get("pca_reconstruction_metrics")
pca_top = csvs.get("pca_top_loadings")
pca_loadings = csvs.get("pca_component_loadings")

selected_pca = pca_cfg.get("selected_pca_components")
achieved_var = pca_cfg.get("achieved_cumulative_variance")

summary_rows = []
if selected_pca is not None:
    summary_rows.append(("selected PCA components", selected_pca))
if achieved_var is not None:
    summary_rows.append(("achieved cumulative variance", achieved_var))
if pca_recon is not None and selected_pca is not None:
    selected_recon = pca_recon.loc[pca_recon["n_components"].eq(selected_pca)]
    if not selected_recon.empty:
        summary_rows.append(("reconstruction RMSE at selected k", selected_recon["reconstruction_rmse"].iloc[0]))
        summary_rows.append(("reconstruction MSE at selected k", selected_recon["reconstruction_mse"].iloc[0]))

if summary_rows:
    display(pd.DataFrame(summary_rows, columns=["metric", "value"]))
else:
    display(Markdown("**PCA summary unavailable:** missing config or reconstruction artifacts."))

if pca_var is not None:
    display(Markdown("### PCA explained variance by component"))
    display(pca_var)

if pca_recon is not None:
    display(Markdown("### PCA reconstruction metrics"))
    display(pca_recon)

,metric,value
0,selected PCA components,8.0000
1,achieved cumulative variance,0.9316
2,reconstruction RMSE at selected k,0.2615
3,reconstruction MSE at selected k,0.0684


### PCA explained variance by component

,component,explained_variance,explained_variance_ratio,cumulative_explained_variance_ratio
0,1,5.6093,0.3740,0.3740
1,2,2.2389,0.1493,0.5232
2,3,1.8164,0.1211,0.6443
3,4,1.3269,0.0885,0.7328
4,5,1.0928,0.0729,0.8056
5,6,0.8298,0.0553,0.8609
6,7,0.5788,0.0386,0.8995
7,8,0.4814,0.0321,0.9316
8,9,0.3411,0.0227,0.9544
9,10,0.2281,0.0152,0.9696


### PCA reconstruction metrics

,n_components,cumulative_explained_variance_ratio,reconstruction_mse,reconstruction_rmse
0,1,0.3740,0.6259,0.7911
1,2,0.5232,0.4767,0.6904
2,3,0.6443,0.3556,0.5963
3,4,0.7328,0.2672,0.5169
4,5,0.8056,0.1943,0.4408
5,6,0.8609,0.1390,0.3729
6,7,0.8995,0.1005,0.3170
7,8,0.9316,0.0684,0.2615
8,9,0.9544,0.0456,0.2136
9,10,0.9696,0.0304,0.1744


## 3. PCA component loadings interpretation

PCA components are linear combinations of the original numeric features. Loadings with larger absolute values contribute more strongly to a component.

Important interpretation cautions:

- component signs are arbitrary;
- a component label is a human summary, not a ground truth category;
- mixed components should be described as mixed rather than forced into a neat label.

In [3]:
def infer_pca_label(pos_features: list[str], neg_features: list[str]) -> str:
    all_features = " ".join(pos_features + neg_features).lower()
    nutrition_terms = ["calories", "fat", "protein", "sodium", "cholesterol", "carbohydrate", "fiber", "sugar"]
    time_terms = ["time", "cooktime", "preptime", "totaltime"]
    count_terms = ["numingredients", "numquantities"]
    serving_terms = ["servings"]

    if sum(term in all_features for term in nutrition_terms) >= 5:
        return "tentative: high overall nutrition density / macro-nutrient axis"
    if any(term in all_features for term in time_terms) and any(term in all_features for term in count_terms + serving_terms):
        return "tentative: time, effort, and recipe scale axis"
    if "sugar" in all_features or "carbohydrate" in all_features:
        if "protein" in all_features or "fat" in all_features or "cholesterol" in all_features:
            return "tentative: carbohydrate/sugar versus protein/fat contrast"
        return "tentative: sweetness/carbohydrate axis"
    if any(term in all_features for term in count_terms):
        return "tentative: ingredient/quantity complexity axis"
    if any(term in all_features for term in serving_terms):
        return "tentative: serving scale axis"
    return "tentative: mixed or unclear numeric contrast"

def pca_component_table(component: int, n: int = 6) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    if pca_loadings is None:
        return pd.DataFrame(), pd.DataFrame(), "missing PCA loadings"
    comp = pca_loadings.loc[pca_loadings["component"].eq(component)].copy()
    if comp.empty:
        return pd.DataFrame(), pd.DataFrame(), "component unavailable"
    pos = comp.sort_values("loading", ascending=False).head(n)
    neg = comp.sort_values("loading", ascending=True).head(n)
    label = infer_pca_label(pos["feature"].astype(str).tolist(), neg["feature"].astype(str).tolist())
    cols = ["feature", "loading", "abs_loading", "explained_variance_ratio"]
    return pos[cols], neg[cols], label

pca_interpretations = []
if selected_pca is None:
    display(Markdown("**Cannot interpret PCA components:** selected component count is missing."))
else:
    for component in range(1, int(selected_pca) + 1):
        pos, neg, label = pca_component_table(component, n=6)
        display(Markdown(f"### PCA component {component}: {label}"))
        display(Markdown("Top positive loadings"))
        display(pos)
        display(Markdown("Top negative loadings"))
        display(neg)
        pca_interpretations.append({
            "component": component,
            "tentative_label": label,
            "top_positive_features": ", ".join(pos["feature"].astype(str).tolist()) if not pos.empty else "unavailable",
            "top_negative_features": ", ".join(neg["feature"].astype(str).tolist()) if not neg.empty else "unavailable",
        })

pca_interpretations_df = pd.DataFrame(pca_interpretations)
if not pca_interpretations_df.empty:
    display(Markdown("### PCA interpretation summary"))
    display(pca_interpretations_df)

### PCA component 1: tentative: high overall nutrition density / macro-nutrient axis

Top positive loadings

,feature,loading,abs_loading,explained_variance_ratio
0,Calories,0.3663,0.3663,0.3740
1,FatContent,0.3480,0.3480,0.3740
8,ProteinContent,0.3351,0.3351,0.3740
2,SaturatedFatContent,0.3278,0.3278,0.3740
4,SodiumContent,0.3105,0.3105,0.3740
3,CholesterolContent,0.2934,0.2934,0.3740


Top negative loadings

,feature,loading,abs_loading,explained_variance_ratio
14,ResolvedServings_imputed,0.0086,0.0086,0.3740
10,PrepTime_Minutes_imputed,0.1305,0.1305,0.3740
7,SugarContent,0.1593,0.1593,0.3740
9,CookTime_Minutes_imputed,0.1847,0.1847,0.3740
11,TotalTime_Minutes_imputed,0.1954,0.1954,0.3740
6,FiberContent,0.2257,0.2257,0.3740


...

,feature,loading,abs_loading,explained_variance_ratio
109,SodiumContent,0.5970,0.5970,0.0321
112,SugarContent,0.3041,0.3041,0.0321
108,CholesterolContent,0.2194,0.2194,0.0321
113,ProteinContent,0.1412,0.1412,0.0321
110,CarbohydrateContent,0.1229,0.1229,0.0321
115,PrepTime_Minutes_imputed,0.0547,0.0547,0.0321


Top negative loadings

,feature,loading,abs_loading,explained_variance_ratio
111,FiberContent,-0.4512,0.4512,0.0321
106,FatContent,-0.3725,0.3725,0.0321
107,SaturatedFatContent,-0.3332,0.3332,0.0321
114,CookTime_Minutes_imputed,-0.0674,0.0674,0.0321
119,ResolvedServings_imputed,-0.0573,0.0573,0.0321
117,NumIngredients,-0.0367,0.0367,0.0321


### PCA interpretation summary

,component,tentative_label,top_positive_features,top_negative_features
0,1,tentative: high overall nutrition density / ma...,"Calories, FatContent, ProteinContent, Saturate...","ResolvedServings_imputed, PrepTime_Minutes_imp..."
1,2,tentative: high overall nutrition density / ma...,"TotalTime_Minutes_imputed, PrepTime_Minutes_im...","SaturatedFatContent, Calories, FatContent, Pro..."
2,3,tentative: high overall nutrition density / ma...,"SugarContent, CarbohydrateContent, FiberConten...","CholesterolContent, SaturatedFatContent, Prote..."
3,4,tentative: high overall nutrition density / ma...,"NumIngredients, NumQuantities, FiberContent, S...","ResolvedServings_imputed, TotalTime_Minutes_im..."
4,5,tentative: high overall nutrition density / ma...,"ResolvedServings_imputed, SaturatedFatContent,...","CookTime_Minutes_imputed, FiberContent, TotalT..."
5,6,tentative: high overall nutrition density / ma...,"PrepTime_Minutes_imputed, FiberContent, TotalT...","CookTime_Minutes_imputed, ResolvedServings_imp..."
6,7,tentative: high overall nutrition density / ma...,"ResolvedServings_imputed, SodiumContent, Fiber...","SugarContent, CholesterolContent, NumQuantitie..."
7,8,tentative: high overall nutrition density / ma...,"SodiumContent, SugarContent, CholesterolConten...","FiberContent, FatContent, SaturatedFatContent,..."


## 4. TruncatedSVD retained energy summary

TruncatedSVD was applied to the sparse TF-IDF content matrix. Its explained variance ratio is best understood as retained energy in a high-dimensional sparse content space. It should not be interpreted as directly comparable to PCA's compact numeric variance structure.

In [4]:
svd_cfg = configs.get("svd_selected_config", {})
svd_sweep = csvs.get("svd_component_sweep")
svd_var = csvs.get("svd_explained_variance_selected")
svd_top = csvs.get("svd_top_terms")

selected_svd = svd_cfg.get("selected_svd_components")
retained_energy = svd_cfg.get("selected_retained_energy")

svd_summary_rows = []
if selected_svd is not None:
    svd_summary_rows.append(("selected SVD components", selected_svd))
if retained_energy is not None:
    svd_summary_rows.append(("selected retained energy", retained_energy))
if svd_cfg.get("reconstruction_sample_size") is not None:
    svd_summary_rows.append(("reconstruction sample size", svd_cfg.get("reconstruction_sample_size")))
if svd_cfg.get("reconstruction_note"):
    svd_summary_rows.append(("reconstruction note", svd_cfg.get("reconstruction_note")))

if svd_summary_rows:
    display(pd.DataFrame(svd_summary_rows, columns=["metric", "value"]))
else:
    display(Markdown("**SVD summary unavailable:** missing config artifact."))

if svd_sweep is not None:
    display(Markdown("### SVD component sweep"))
    display(svd_sweep)

if svd_var is not None:
    display(Markdown("### Selected SVD cumulative energy"))
    display(svd_var.head(30))

,metric,value
0,selected SVD components,200
1,selected retained energy,0.5274
2,reconstruction sample size,5379
3,reconstruction note,Reconstruction evaluated on 5379 sampled rows;...


### SVD component sweep

,n_components,retained_energy,reconstruction_mse_sample,reconstruction_rmse_sample,input_shape_rows,input_shape_cols,output_shape_rows,output_shape_cols,random_state,runtime_seconds
0,50,0.2830,0.0002,0.0149,522517,4647,522517,50,42,2.2860
1,100,0.3944,0.0002,0.0137,522517,4647,522517,100,42,4.4883
2,200,0.5274,0.0001,0.0121,522517,4647,522517,200,42,8.6052
3,300,0.6125,0.0001,0.0110,522517,4647,522517,300,42,13.3838


### Selected SVD cumulative energy

,component,explained_variance,explained_variance_ratio,cumulative_explained_variance_ratio
0,1,0.0118,0.0082,0.0082
1,2,0.0350,0.0244,0.0326
2,3,0.0242,0.0168,0.0494
3,4,0.0197,0.0137,0.0631
4,5,0.0154,0.0107,0.0738
5,6,0.0137,0.0095,0.0833
6,7,0.0120,0.0083,0.0917
7,8,0.0108,0.0075,0.0992
8,9,0.0105,0.0073,0.1065
9,10,0.0101,0.0070,0.1135


## 5. TruncatedSVD latent content themes

SVD components are latent semantic axes learned from weighted ingredient, keyword, category, and yield-unit tokens. The positive and negative sides are not inherently good or bad; the sign can flip without changing the model's meaning.

The labels below are tentative summaries based on top terms. Components with broad high-frequency tags or mixed terms should be treated cautiously.

In [5]:
def infer_svd_theme(pos_terms: list[str], neg_terms: list[str]) -> str:
    terms = " ".join(pos_terms + neg_terms).lower()
    if any(x in terms for x in ["dessert", "sugar", "butter", "flour", "cake", "cookie", "chocolate"]):
        return "tentative: baking/dessert or sweet recipe theme"
    if any(x in terms for x in ["meat", "poultry", "chicken", "beef", "pork", "onion", "garlic"]):
        return "tentative: meat/main-dish or savory cooking theme"
    if any(x in terms for x in ["asian", "soy", "ginger", "curry", "indian", "thai", "chinese"]):
        return "tentative: Asian or spice-forward flavor profile"
    if any(x in terms for x in ["healthy", "low_cholesterol", "low_protein", "vegan", "vegetable"]):
        return "tentative: health-oriented / vegetable theme"
    if any(x in terms for x in ["easy", "beginner_cook", "inexpensive", "quick"]):
        return "tentative: quick/easy recipe tag theme"
    if any(x in terms for x in ["beverage", "drink", "smoothie"]):
        return "tentative: beverage theme"
    return "tentative: mixed or unclear content theme"

def svd_component_terms(component: int, n: int = 8) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    if svd_top is None:
        return pd.DataFrame(), pd.DataFrame(), "missing SVD top terms"
    comp = svd_top.loc[svd_top["component"].eq(component)].copy()
    if comp.empty:
        return pd.DataFrame(), pd.DataFrame(), "component unavailable"
    pos = comp.loc[comp["side"].eq("positive")].sort_values("rank").head(n)
    neg = comp.loc[comp["side"].eq("negative")].sort_values("rank").head(n)
    label = infer_svd_theme(pos["feature_name"].astype(str).tolist(), neg["feature_name"].astype(str).tolist())
    cols = ["rank", "feature_name", "feature_group", "loading", "abs_loading"]
    return pos[cols], neg[cols], label

max_components_to_show = 20
if selected_svd is not None:
    max_components_to_show = min(max_components_to_show, int(selected_svd))

svd_interpretations = []
for component in range(1, max_components_to_show + 1):
    pos, neg, label = svd_component_terms(component, n=8)
    display(Markdown(f"### SVD component {component}: {label}"))
    display(Markdown("Top positive terms"))
    display(pos)
    display(Markdown("Top negative terms"))
    display(neg)
    svd_interpretations.append({
        "component": component,
        "tentative_label": label,
        "top_positive_terms": ", ".join(pos["feature_name"].astype(str).tolist()) if not pos.empty else "unavailable",
        "top_negative_terms": ", ".join(neg["feature_name"].astype(str).tolist()) if not neg.empty else "unavailable",
    })

svd_interpretations_df = pd.DataFrame(svd_interpretations)
if not svd_interpretations_df.empty:
    display(Markdown("### SVD interpretation summary"))
    display(svd_interpretations_df)

### SVD component 1: tentative: baking/dessert or sweet recipe theme

Top positive terms

,rank,feature_name,feature_group,loading,abs_loading
0,1,keyword__easy,keywords,0.5040,0.5040
1,2,category__dessert,category,0.2458,0.2458
2,3,ingredient__salt,ingredients,0.2240,0.2240
3,4,ingredient__butter,ingredients,0.2146,0.2146
4,5,ingredient__sugar,ingredients,0.1997,0.1997
5,6,ingredient__eggs,ingredients,0.1687,0.1687
6,7,keyword__healthy,keywords,0.1526,0.1526
7,8,ingredient__flour,ingredients,0.1480,0.1480


Top negative terms

,rank,feature_name,feature_group,loading,abs_loading
15,1,category__ham_and_bean_soup,category,0.0000,0.0000
16,2,category__key_lime_pie,category,0.0000,0.0000
17,3,category__fish_tuna,category,0.0000,0.0000
18,4,category__dairy_free_foods,category,0.0000,0.0000
19,5,category__mushroom_soup,category,0.0000,0.0000
20,6,category__snacks_sweet,category,0.0000,0.0000
21,7,category__wheat_bread,category,0.0000,0.0000
22,8,category__black_bean_soup,category,0.0000,0.0000


...

,rank,feature_name,feature_group,loading,abs_loading
570,1,keyword__oven,keywords,0.3460,0.3460
571,2,keyword__beginner_cook,keywords,0.2138,0.2138
572,3,keyword__european,keywords,0.2079,0.2079
573,4,keyword__high_in,keywords,0.1821,0.1821
574,5,ingredient__flour,ingredients,0.1726,0.1726
575,6,keyword__healthy,keywords,0.1623,0.1623
576,7,ingredient__parmesan_cheese,ingredients,0.1443,0.1443
577,8,ingredient__olive_oil,ingredients,0.1373,0.1373


Top negative terms

,rank,feature_name,feature_group,loading,abs_loading
585,1,category__vegetable,category,-0.2696,0.2696
586,2,category__one_dish_meal,category,-0.2550,0.2550
587,3,category__lunch_snacks,category,-0.2409,0.2409
588,4,ingredient__all_purpose_flour,ingredients,-0.2346,0.2346
589,5,keyword__low_protein,keywords,-0.2040,0.2040
590,6,ingredient__milk,ingredients,-0.1998,0.1998
591,7,ingredient__vanilla_extract,ingredients,-0.1403,0.1403
592,8,keyword__asian,keywords,-0.1320,0.1320


### SVD interpretation summary

,component,tentative_label,top_positive_terms,top_negative_terms
0,1,tentative: baking/dessert or sweet recipe theme,"keyword__easy, category__dessert, ingredient__...","category__ham_and_bean_soup, category__key_lim..."
1,2,tentative: baking/dessert or sweet recipe theme,"keyword__easy, keyword__meat, ingredient__onio...","category__dessert, ingredient__sugar, keyword_..."
2,3,tentative: baking/dessert or sweet recipe theme,"keyword__easy, category__dessert, category__be...","keyword__meat, keyword__poultry, category__one..."
3,4,tentative: baking/dessert or sweet recipe theme,"keyword__low_cholesterol, keyword__healthy, ke...","keyword__meat, keyword__poultry, keyword__easy..."
4,5,tentative: baking/dessert or sweet recipe theme,"category__dessert, keyword__meat, keyword__pou...","keyword__dessert, keyword__vegetable, category..."
5,6,tentative: baking/dessert or sweet recipe theme,"keyword__vegetable, category__dessert, keyword...","keyword__dessert, keyword__meat, keyword__poul..."
6,7,tentative: baking/dessert or sweet recipe theme,"keyword__dessert, keyword__cookie_and_brownie,...","keyword__breads, ingredient__baking_powder, in..."
7,8,tentative: baking/dessert or sweet recipe theme,"category__lunch_snacks, keyword__beginner_cook...","keyword__cookie_and_brownie, category__vegetab..."
8,9,tentative: baking/dessert or sweet recipe theme,"category__lunch_snacks, keyword__for_large_gro...","ingredient__milk, category__one_dish_meal, ing..."
9,10,tentative: baking/dessert or sweet recipe theme,"category__one_dish_meal, keyword__inexpensive,...","keyword__vegetable, ingredient__butter, keywor..."


## 6. PCA vs SVD comparison

The dimensionality comparison table summarizes why the two reduction methods behave differently. PCA acts on a small dense matrix of standardized numeric features, while SVD acts on a much larger sparse content matrix with many diverse tokens.

In [6]:
dim = csvs.get("dimensionality_comparison")
if dim is not None:
    display(dim)
else:
    display(Markdown("**Dimensionality comparison unavailable.**"))

if selected_pca is not None and achieved_var is not None and selected_svd is not None and retained_energy is not None:
    display(Markdown(f"""
**Comparison note.** PCA reached {achieved_var:.4f} cumulative explained variance with {selected_pca} numeric components because the dense numeric matrix has only 15 standardized features and meaningful redundancy among nutrition, time, complexity, and servings. TruncatedSVD retained {retained_energy:.4f} energy with {selected_svd} content components because sparse text/list/category features are more diverse and diffuse across thousands of possible tokens. This lower retained energy is expected and does not mean the content embedding failed.
"""))

,representation,input_shape,output_shape,original_dimensions,reduced_dimensions,method,sparse_or_dense,preprocessing_source,selected_components,variance_or_energy_retained,reconstruction_mse,reconstruction_rmse,intended_downstream_use,notes
0,numeric_scaled,"(522517, 15)","(522517, 15)",15,15,StandardScaler output,dense,Step 3 numeric matrix,NaN,NaN,NaN,NaN,Input to numeric PCA.,Dense scaled numeric matrix after clipping/log...
1,numeric_pca_selected,"(522517, 15)","(522517, 8)",15,8,PCA,dense,Step 3 numeric matrix,8.0000,0.9316,0.0684,0.2615,Compact numeric recipe structure.,PCA-selected numeric representation using 90% ...
2,content_tfidf,"(522517, 4647)","(522517, 4647)",4647,4647,TF-IDF sparse matrix,sparse,Step 4 content TF-IDF matrix,NaN,NaN,NaN,NaN,Input to content SVD.,"Sparse TF-IDF matrix from ingredients, keyword..."
3,content_svd_selected,"(522517, 4647)","(522517, 200)",4647,200,TruncatedSVD,dense,Step 4 content TF-IDF matrix,200.0000,0.5274,0.0001,0.0121,Dense content embedding.,TruncatedSVD dense content embedding; retained...
4,combined_recipe_reduced,"(522517, 200) + (522517, 8)","(522517, 208)",4662,208,Concatenation,dense,Step 5 PCA/SVD outputs,208.0000,NaN,NaN,NaN,Future clustering and content-based recommenda...,Concatenated content SVD and numeric PCA repre...



**Comparison note.** PCA reached 0.9316 cumulative explained variance with 8 numeric components because the dense numeric matrix has only 15 standardized features and meaningful redundancy among nutrition, time, complexity, and servings. TruncatedSVD retained 0.5274 energy with 200 content components because sparse text/list/category features are more diverse and diffuse across thousands of possible tokens. This lower retained energy is expected and does not mean the content embedding failed.


## 7. Report-ready interpretation notes

The following cell creates concise paragraphs that can be adapted into the Week 5 report. They are generated from the loaded artifacts where possible and avoid inventing values when artifacts are missing.

In [7]:
def fmt(value, digits=4):
    if value is None:
        return "TODO: run script and fill from artifact"
    try:
        return f"{float(value):.{digits}f}"
    except Exception:
        return str(value)

pca_para = (
    f"PCA was applied to the dense scaled numeric matrix. The selected representation uses "
    f"{selected_pca if selected_pca is not None else 'TODO: run script and fill from artifact'} components, "
    f"which preserves {fmt(achieved_var)} cumulative explained variance. The selected reconstruction RMSE is "
    f"{fmt(selected_recon['reconstruction_rmse'].iloc[0]) if 'selected_recon' in globals() and not selected_recon.empty else 'TODO: run script and fill from artifact'}. "
    "The early components are interpreted tentatively from loadings: the first component is dominated by broad nutrition magnitude, "
    "while later components capture time/effort, carbohydrate/sugar contrasts, and recipe complexity."
)

svd_para = (
    f"TruncatedSVD was applied to the sparse content TF-IDF matrix. The selected model uses "
    f"{selected_svd if selected_svd is not None else 'TODO: run script and fill from artifact'} components and retains "
    f"{fmt(retained_energy)} energy. The retained energy is lower than numeric PCA because ingredient, keyword, category, "
    "and yield-unit content is sparse and semantically diffuse. Component labels are tentative latent themes inferred from top terms, "
    "not observed recipe categories."
)

comparison_para = (
    "PCA and TruncatedSVD serve different representation problems. PCA compresses a small dense numeric matrix where scaled features "
    "share covariance structure, so a small number of components can retain most numeric variance. TruncatedSVD compresses a high-dimensional "
    "sparse content matrix, where useful information is distributed across many low-frequency semantic tokens. The combined reduced matrix joins "
    "content SVD features with numeric PCA features for future clustering and content-based recommendation experiments."
)

report_notes = {
    "PCA interpretation": pca_para,
    "SVD interpretation": svd_para,
    "PCA vs SVD comparison": comparison_para,
}

for title, paragraph in report_notes.items():
    display(Markdown(f"### {title}\n\n{paragraph}"))

### PCA interpretation

PCA was applied to the dense scaled numeric matrix. The selected representation uses 8 components, which preserves 0.9316 cumulative explained variance. The selected reconstruction RMSE is 0.2615. The early components are interpreted tentatively from loadings: the first component is dominated by broad nutrition magnitude, while later components capture time/effort, carbohydrate/sugar contrasts, and recipe complexity.

### SVD interpretation

TruncatedSVD was applied to the sparse content TF-IDF matrix. The selected model uses 200 components and retains 0.5274 energy. The retained energy is lower than numeric PCA because ingredient, keyword, category, and yield-unit content is sparse and semantically diffuse. Component labels are tentative latent themes inferred from top terms, not observed recipe categories.

### PCA vs SVD comparison

PCA and TruncatedSVD serve different representation problems. PCA compresses a small dense numeric matrix where scaled features share covariance structure, so a small number of components can retain most numeric variance. TruncatedSVD compresses a high-dimensional sparse content matrix, where useful information is distributed across many low-frequency semantic tokens. The combined reduced matrix joins content SVD features with numeric PCA features for future clustering and content-based recommendation experiments.

### Optional export: component interpretation notes

This cell exports the report-ready paragraphs and the component interpretation summary tables to `artifacts/week5/pca_svd/component_interpretation_notes.md`. It writes only a Markdown notes file and does not modify matrices or models.

In [ ]:
notes_lines = ["# Week 5 PCA/SVD Component Interpretation Notes", ""]
for title, paragraph in report_notes.items():
    notes_lines.extend([f"## {title}", "", paragraph, ""])

if 'pca_interpretations_df' in globals() and not pca_interpretations_df.empty:
    notes_lines.extend(["## PCA component interpretation summary", "", '```text\n' + pca_interpretations_df.to_string(index=False) + '\n```', ""])

if 'svd_interpretations_df' in globals() and not svd_interpretations_df.empty:
    notes_lines.extend(["## SVD component interpretation summary", "", '```text\n' + svd_interpretations_df.to_string(index=False) + '\n```', ""])

NOTES_OUT.parent.mkdir(parents=True, exist_ok=True)
NOTES_OUT.write_text("\n".join(notes_lines), encoding="utf-8")
print(f"Wrote notes to {NOTES_OUT}")

## 8. Open questions / caveats

- PCA component labels are tentative because components are linear algebra constructs, not human-defined categories.
- SVD signs are arbitrary; positive and negative sides should be interpreted as opposite directions on a latent axis, not good versus bad.
- Very broad tags such as `keyword__easy` can dominate early SVD components because they occur often.
- Sparse text retained energy is expected to be lower than numeric PCA variance because content tokens are diverse and diffuse.
- These interpretations should be validated later through recipe examples, cluster profiles, or recommendation case studies.
- This notebook does not evaluate recommender quality.